# Practice: Cleaning and Aggregating Session Data

In this notebook we practice the most common DataFrame operations on a small dataset of user sessions (`data/misc/session_info.csv`): cleaning, new columns, aggregations, joins and caching.

The file is separated by `;` instead of `,`, and some addresses span two lines, so we need to tell the reader about both.

In [ ]:
from pyspark.sql.functions import col, count, lower, when, concat, lit, length, regexp_extract, avg, max, min, sum, desc

## Setup and loading

In [ ]:
import os
from pyspark.sql import SparkSession

# On Databricks this returns the session that already exists.
# Locally (pip install pyspark) or on Colab it starts a new one.
spark = SparkSession.builder.getOrCreate()

# display(df) shows an interactive table only on Databricks: elsewhere we use show() instead
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    def display(df):
        df.show(20, truncate=False)

We read the data directly from the GitHub repository of the course, so there is nothing to upload. pandas downloads the CSV and Spark turns it into a DataFrame. This is fine for small files; with real big data Spark would read the files directly from distributed storage (see [Lesson 7](https://github.com/Naviden/Introduction-to-Big-Data-and-NoSQL/blob/main/lessons/07_spark_basics_part1.md) for the other options).

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data"

def load_csv(path, **kwargs):
    """Read a CSV file of the course repository and return a Spark DataFrame."""
    pdf = pd.read_csv(f"{DATA_URL}/{path}", **kwargs)
    pdf = pdf.astype(object).where(pdf.notna(), None)  # pandas NaN -> Spark null
    return spark.createDataFrame(pdf)

In [ ]:
# sep=";" because the columns are separated by semicolons
df = load_csv("misc/session_info.csv", sep=";")

If you uploaded the file to a volume in Databricks, the equivalent Spark reader needs `multiLine=True`, because some addresses contain a line break inside the quotes:

```python
df = spark.read.csv("/Volumes/workspace/default/course_data/session_info.csv",
                    header=True, inferSchema=True, sep=";", multiLine=True)
```

## Initial data inspection

In [ ]:
print("Initial Schema:")
df.printSchema()

In [ ]:
print("Initial Data:")
df.show(5, truncate=False)

## Data cleaning and preprocessing

In [ ]:
cleaned_df = df \
    .withColumn("user_name", lower(col("user_name"))) \
    .withColumn("email_domain", regexp_extract(col("email"), r'@(.+)', 1)) \
    .filter(col("email").isNotNull()) \
    .dropDuplicates(["user_id"]) \
    .withColumn("address_length", length(col("address")))

## Transformations and new columns

In [ ]:
transformed_df = cleaned_df \
    .withColumn("status", when(col("address_length") > 50, lit("Detailed")).otherwise(lit("Short"))) \
    .withColumn("full_user_info", concat(col("user_name"), lit(" - "), col("email_domain")))

## Aggregations

In [ ]:
user_stats = cleaned_df.groupBy("user_name").agg(
    count("*").alias("session_count"),
    avg("address_length").alias("avg_address_length"),
    max("address_length").alias("max_address_length"),
    min("address_length").alias("min_address_length")
)

In [ ]:
# Sorting
sorted_user_stats = user_stats.orderBy(desc("session_count"))

## Joins

In [ ]:
joined_df = transformed_df.join(user_stats, on="user_name", how="inner")

## Caching
`cache()` asks Spark to keep a DataFrame in memory after the first time it is computed, so the next actions on it don't recompute everything from the file. It is useful when you use the same DataFrame many times.

> On Databricks Free Edition (serverless compute) `cache()` is not supported and raises an error, so we skip it there.

In [ ]:
try:
    joined_df.cache()
except Exception as e:
    print("cache() not available here:", type(e).__name__)

print(f"Rows in joined_df: {joined_df.count()}")

## Grouped aggregations and additional insights

In [ ]:
email_domain_stats = cleaned_df.groupBy("email_domain").agg(
    count("user_id").alias("user_count"),
    sum("address_length").alias("total_address_length")
).orderBy(desc("user_count"))

## Data insights

In [ ]:
print("Transformed Data:")
transformed_df.show(5, truncate=False)

print("User Statistics:")
sorted_user_stats.show(5, truncate=False)

print("Joined Data:")
joined_df.show(5, truncate=False)

print("Email Domain Statistics:")
email_domain_stats.show(5, truncate=False)

## What about `spark.stop()`?

The `spark.stop()` method shuts down a SparkSession and releases its resources (memory, CPU, executors).

- **In a standalone script or a local session** (for example `pip install pyspark` on your computer), call `spark.stop()` at the end: it releases the resources and lets other Spark jobs start with a clean slate. It is similar to closing a file after opening it.
- **On Databricks, do not call it.** The session is created and managed by Databricks and shared by all the cells of the notebook (and, on a shared cluster, by other notebooks too). Stopping it breaks the following cells until the notebook is detached and re-attached.

```python
# Only when running Spark yourself, outside Databricks:
spark.stop()
```

## Exercises
1. Extract the hour from `timestamp` and count the sessions per hour of the day.
2. Which email domain has the longest average address?
3. Add a column `state` with the two-letter US state code at the end of the address (hint: `regexp_extract` with the pattern `([A-Z]{2}) \d{5}$`).